[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/25.%20The%20Dynamic%20Truck%20Appointment%20System%20Problem/P25-Tier-2_executed.ipynb)

---


# 25. The Dynamic Truck Appointment System Problem
## Tier 2: The Classic Heuristic (Priority-Based Dynamic Scheduling)

### Goal and scope

This notebook demonstrates a greedy, priority-based heuristic for adjusting truck appointments after delay information becomes available. The input is a set of trucks with original appointment slots, expected delays, and urgency weights, together with a set of capacity-limited time slots. The output is a revised assignment, a decision log, cost estimates, and visual summaries.

The method processes one snapshot of delay updates: it ranks trucks once, then assigns them one at a time. It is a heuristic, not an exact optimization method, and it does not guarantee a globally minimum-cost schedule. Repeated updates would require rerunning the scheduling procedure with the latest information.

### Model assumptions

- Each truck occupies exactly one appointment slot, and each slot has a fixed capacity.
- Slot IDs are ordered in time; a larger ID denotes a later slot. In this example, adjacent slots are 30 minutes apart.
- A later appointment can absorb up to 10 minutes of a truck's expected delay per slot moved later. Earlier appointments do not absorb delay in this model.
- Waiting cost is proportional to the delay that remains after this absorption. A change from the original appointment incurs a fixed rescheduling cost.
- Priority weights and score coefficients are heuristic design choices, not probabilities or learned estimates. The model omits factors such as travel time, arrival-time distributions, service durations, and interactions between different trucks' waiting times.

### How the heuristic works

A priority score determines the order in which trucks are considered. For each truck, the algorithm compares its current slot with available alternatives using a slot score. A move is made only when an alternative has a strictly higher score. Once a truck is assigned, its slot's occupancy is updated before the next truck is considered; therefore, later decisions reflect earlier assignments.

The priority score for truck $i$ is

$$
P_i = 2d_i + 15u_i + 5(13-s_i),
$$

where $d_i$ is its expected delay in minutes, $u_i$ is its urgency weight, and $s_i$ is its current slot ID when the priority queue is created. Larger scores are processed first. The score is a ranking rule; it is not the cost objective.

For a truck with original slot $o_i$, candidate slot $s$, expected delay $d_i$, waiting-cost rate $c_w$, and rescheduling cost $c_r$, define the delay absorbed by moving later as

$$
r_i(s) = \min\left(d_i,\;10\max(0,s-o_i)\right).
$$

The slot score used by the heuristic is

$$
Q_i(s) = c_w r_i(s)
- c_r\,\mathbf{1}[s \ne o_i]
+ 20\,\mathbf{1}[\text{slot }s\text{ has spare capacity}]
- 5\,L_s,
$$

where $L_s$ is the number of trucks currently assigned to slot $s$. The indicator $\mathbf{1}[\cdot]$ equals 1 when its condition is true and 0 otherwise. The availability bonus and load penalty are heuristic tie-breaking and balancing terms; they are distinct from the monetary cost calculation. The implementation is in `calculate_slot_score`, and the greedy comparison is in `find_best_slot`.

The reported system cost is

$$
C = \sum_i c_w\left(d_i-r_i(s_i)\right)
+ c_r\sum_i \mathbf{1}[s_i \ne o_i],
$$

where $s_i$ is the final slot assigned to truck $i$. The first term is waiting cost; the second is rescheduling cost. `SchedulingSystem.calculate_total_cost` implements this expression. Static scheduling is the reference case $s_i=o_i$ for every truck, so it incurs no rescheduling cost and retains each truck's full expected delay.

### Algorithm and computational effort

The priority queue orders $n$ trucks by their initial priority scores. For each truck, the algorithm scans $m$ slots, so slot selection takes $O(nm)$ score evaluations. Heap insertion and removal take $O(n\log n)$ in total. Updating assignment lists can require up to $O(C)$ work per move, where $C$ is the largest slot capacity, because removing an item from a Python list is linear in its length. Thus a useful upper-bound description is $O(n\log n + nm + nC)$, with $O(n+m)$ storage apart from the assignment lists and log. Runtime depends on the implementation and hardware; this complexity statement is not a promise of millisecond execution.

### What to examine

- The priority scores and the order in which trucks are considered.
- Which trucks change slots, and how slot capacity affects later decisions.
- Waiting and rescheduling costs separately, as well as their sum.
- The original and final assignments, and measured runtime for the generated scalability cases.

The heuristic favors fast, interpretable decisions over an optimality guarantee. Its results depend on the priority formula, slot-score coefficients, capacity limits, and the delay-absorption assumption.

In [ ]:
# These packages support the scheduling model, its data tables, and its figures.
import heapq
import sys
import time
from dataclasses import dataclass, field
from typing import Any, Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
@dataclass
class Truck:
    """Store a truck's original and current appointment information."""

    id: str
    original_slot: int
    current_slot: int
    expected_delay: int  # Minutes
    urgency_weight: float = 1.0
    cost_weight: float = 1.0

    @property
    def priority_score(self) -> float:
        """Return the heuristic score used to order trucks."""
        delay_score = self.expected_delay * 2.0
        urgency_score = self.urgency_weight * 15.0
        slot_score = (13 - self.current_slot) * 5.0
        return delay_score + urgency_score + slot_score


@dataclass
class TimeSlot:
    """Store a time slot's capacity and current truck assignments."""

    slot_id: int
    start_time: str
    capacity: int
    current_assignments: List[str] = field(default_factory=list)

    def add_truck(self, truck_id: str) -> bool:
        """Add a truck only when the slot has spare capacity."""
        if self.is_available:
            self.current_assignments.append(truck_id)
            return True
        return False

    def remove_truck(self, truck_id: str) -> bool:
        """Remove a truck and report whether it was assigned here."""
        if truck_id in self.current_assignments:
            self.current_assignments.remove(truck_id)
            return True
        return False

    @property
    def is_available(self) -> bool:
        """Return whether the slot can accept another truck."""
        return len(self.current_assignments) < self.capacity


@dataclass
class SchedulingSystem:
    """Hold the trucks, slots, and cost parameters for one scheduling run."""

    trucks: Dict[str, Truck]
    slots: Dict[int, TimeSlot]
    rescheduling_cost_per_change: float = 50.0
    waiting_cost_per_minute: float = 5.0

    def calculate_total_cost(self) -> Dict[str, float]:
        """Calculate waiting, rescheduling, and combined costs."""
        total_waiting_cost = 0.0
        total_rescheduling_cost = 0.0

        for truck in self.trucks.values():
            absorbed_delay = min(
                truck.expected_delay,
                max(0, truck.current_slot - truck.original_slot) * 10,
            )
            remaining_delay = truck.expected_delay - absorbed_delay
            total_waiting_cost += (
                remaining_delay * self.waiting_cost_per_minute
            )

            if truck.current_slot != truck.original_slot:
                total_rescheduling_cost += self.rescheduling_cost_per_change

        return {
            'waiting_cost': total_waiting_cost,
            'rescheduling_cost': total_rescheduling_cost,
            'total_cost': total_waiting_cost + total_rescheduling_cost,
        }

In [ ]:
def create_example_system() -> SchedulingSystem:
    """Create the five-truck example and its six appointment slots."""
    trucks = {
        'A': Truck('A', 2, 2, 30, 1.0, 1.0),
        'B': Truck('B', 3, 3, 0, 0.5, 1.0),
        'C': Truck('C', 2, 2, 45, 1.5, 1.2),
        'D': Truck('D', 4, 4, 15, 0.8, 1.0),
        'E': Truck('E', 5, 5, 0, 0.6, 1.0),
    }

    slots = {
        1: TimeSlot(1, '09:00', 2),
        2: TimeSlot(2, '09:30', 2),
        3: TimeSlot(3, '10:00', 2),
        4: TimeSlot(4, '10:30', 2),
        5: TimeSlot(5, '11:00', 2),
        6: TimeSlot(6, '11:30', 2),
    }

    for truck in trucks.values():
        if not slots[truck.original_slot].add_truck(truck.id):
            raise ValueError('An original appointment exceeds slot capacity.')

    return SchedulingSystem(trucks, slots)


system = create_example_system()
print('Initial system state:')
for truck_id, truck in system.trucks.items():
    print(
        f'Truck {truck_id}: slot {truck.current_slot}, '
        f'expected delay {truck.expected_delay} minutes, '
        f'priority {truck.priority_score:.1f}'
    )

In [ ]:
def priority_based_dynamic_scheduling(
    system: SchedulingSystem,
) -> Dict[str, Any]:
    """Greedily assign trucks in descending initial-priority order."""
    start_time = time.perf_counter()
    scheduling_log = []

    # Store the priority at queue creation so processing order is a snapshot.
    priority_queue = []
    for truck in system.trucks.values():
        heapq.heappush(
            priority_queue,
            (-truck.priority_score, truck.id, truck),
        )

    while priority_queue:
        neg_priority, _, truck = heapq.heappop(priority_queue)
        initial_priority = -neg_priority
        old_slot = truck.current_slot
        best_slot = find_best_slot(system, truck)

        if best_slot != old_slot:
            removed = system.slots[old_slot].remove_truck(truck.id)
            if not removed:
                raise RuntimeError('Truck is missing from its current slot.')

            added = system.slots[best_slot].add_truck(truck.id)
            if not added:
                raise RuntimeError('The selected slot has no spare capacity.')

            truck.current_slot = best_slot
            reason = 'The available slot has a higher heuristic score.'
        else:
            reason = 'No available alternative has a higher heuristic score.'

        scheduling_log.append(
            {
                'truck': truck.id,
                'priority_score': initial_priority,
                'old_slot': old_slot,
                'new_slot': truck.current_slot,
                'delay': truck.expected_delay,
                'reason': reason,
            }
        )

    execution_time = time.perf_counter() - start_time

    # These checks connect the assignment representation to the capacity model.
    assigned_trucks = [
        truck_id
        for slot in system.slots.values()
        for truck_id in slot.current_assignments
    ]
    assert len(assigned_trucks) == len(system.trucks)
    assert set(assigned_trucks) == set(system.trucks)
    assert all(
        len(slot.current_assignments) <= slot.capacity
        for slot in system.slots.values()
    )
    assert all(
        system.slots[truck.current_slot].current_assignments.count(truck.id) == 1
        for truck in system.trucks.values()
    )

    return {
        'scheduling_log': scheduling_log,
        'execution_time': execution_time,
        'final_costs': system.calculate_total_cost(),
    }


def find_best_slot(system: SchedulingSystem, truck: Truck) -> int:
    """Return the highest-scoring available slot, if it improves the score."""
    best_slot = truck.current_slot
    best_score = calculate_slot_score(system, truck, best_slot)

    for slot_id, slot in system.slots.items():
        if slot_id != truck.current_slot and not slot.is_available:
            continue

        score = calculate_slot_score(system, truck, slot_id)
        if score > best_score:
            best_score = score
            best_slot = slot_id

    return best_slot


def calculate_slot_score(
    system: SchedulingSystem,
    truck: Truck,
    slot_id: int,
) -> float:
    """Score a slot using delay absorption, change cost, and occupancy."""
    slot = system.slots[slot_id]
    delay_reduction = min(
        truck.expected_delay,
        max(0, slot_id - truck.original_slot) * 10,
    )
    delay_benefit = delay_reduction * system.waiting_cost_per_minute

    reschedule_penalty = 0.0
    if slot_id != truck.original_slot:
        reschedule_penalty = system.rescheduling_cost_per_change

    availability_bonus = 20.0 if slot.is_available else 0.0
    load_penalty = len(slot.current_assignments) * 5.0

    return (
        delay_benefit
        - reschedule_penalty
        + availability_bonus
        - load_penalty
    )

In [ ]:
print('Running priority-based dynamic scheduling...')
result = priority_based_dynamic_scheduling(system)

print(
    f"\nAlgorithm completed in "
    f"{result['execution_time'] * 1000:.3f} milliseconds"
)
print('\nFinal costs:')
for cost_type, cost_value in result['final_costs'].items():
    print(f"  {cost_type.replace('_', ' ').title()}: ${cost_value:.2f}")

print('\nScheduling decisions:')
for index, entry in enumerate(result['scheduling_log'], start=1):
    status = 'RESCHEDULED' if entry['old_slot'] != entry['new_slot'] else 'UNCHANGED'
    print(
        f"{index}. Truck {entry['truck']}: slot {entry['old_slot']} "
        f"to slot {entry['new_slot']} "
        f"(priority {entry['priority_score']:.1f}, "
        f"delay {entry['delay']} minutes) [{status}]"
    )

In [ ]:
def calculate_static_scheduling_costs(
    system: SchedulingSystem,
) -> Dict[str, float]:
    """Calculate the reference cost when every truck keeps its original slot."""
    total_waiting_cost = sum(
        truck.expected_delay * system.waiting_cost_per_minute
        for truck in system.trucks.values()
    )

    return {
        'waiting_cost': total_waiting_cost,
        'rescheduling_cost': 0.0,
        'total_cost': total_waiting_cost,
    }


static_costs = calculate_static_scheduling_costs(system)
dynamic_costs = result['final_costs']
assert np.isclose(
    dynamic_costs['total_cost'],
    dynamic_costs['waiting_cost'] + dynamic_costs['rescheduling_cost'],
)

cost_savings = static_costs['total_cost'] - dynamic_costs['total_cost']
if not np.isclose(static_costs['total_cost'], 0.0):
    improvement_percent = cost_savings / static_costs['total_cost'] * 100
else:
    improvement_percent = 0.0

print('Cost comparison:')
print(f"Static scheduling:  ${static_costs['total_cost']:.2f}")
print(f"Dynamic scheduling: ${dynamic_costs['total_cost']:.2f}")
print(f'Savings:            ${cost_savings:.2f}')
print(f'Cost reduction:     {improvement_percent:.1f}%')

In [ ]:
def create_comprehensive_visualizations(
    system: SchedulingSystem,
    result: Dict[str, Any],
    static_costs: Dict[str, float],
) -> None:
    """Plot priority, assignments, costs, and run-specific metrics."""
    dynamic_costs = result['final_costs']
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        'Priority-Based Dynamic Scheduling Analysis',
        fontsize=16,
        fontweight='bold',
    )

    # The log retains the queue's initial priorities, so this chart shows its processing order.
    ax1 = axes[0, 0]
    priority_df = pd.DataFrame(result['scheduling_log'])
    priority_df = priority_df.sort_values(
        'priority_score', ascending=True, kind='stable'
    )
    colors = [
        '#ff7f0e' if delay > 0 else '#1f77b4'
        for delay in priority_df['delay']
    ]
    ax1.barh(priority_df['truck'], priority_df['priority_score'], color=colors)
    ax1.set_xlabel('Priority score (higher is processed first)')
    ax1.set_ylabel('Truck')
    ax1.set_title('Truck Priority Order')
    ax1.grid(True, axis='x', alpha=0.3)
    for position, (_, row) in enumerate(priority_df.iterrows()):
        ax1.text(
            row['priority_score'] + 2,
            position,
            f"{row['delay']} min delay",
            va='center',
            fontsize=9,
        )
    ax1.legend(
        handles=[
            plt.Rectangle((0, 0), 1, 1, color='#ff7f0e', label='Positive expected delay'),
            plt.Rectangle((0, 0), 1, 1, color='#1f77b4', label='No expected delay'),
        ],
        loc='lower right',
    )

    # Distinct markers identify original and final slot assignments.
    ax2 = axes[0, 1]
    timeline_rows = []
    for truck in system.trucks.values():
        timeline_rows.append(
            {
                'truck': truck.id,
                'original_slot': truck.original_slot,
                'final_slot': truck.current_slot,
            }
        )
    timeline_df = pd.DataFrame(timeline_rows)
    for _, row in timeline_df.iterrows():
        ax2.scatter(
            row['original_slot'], row['truck'],
            s=100, color='red', marker='o',
            label='Original assignment' if row.name == 0 else None,
        )
        ax2.scatter(
            row['final_slot'], row['truck'],
            s=100, color='blue', marker='s',
            label='Final assignment' if row.name == 0 else None,
        )
        if row['original_slot'] != row['final_slot']:
            ax2.annotate(
                '',
                xy=(row['final_slot'], row['truck']),
                xytext=(row['original_slot'], row['truck']),
                arrowprops={'arrowstyle': '->', 'color': 'green', 'linewidth': 2},
            )
    ax2.set_xlabel('Appointment slot start time')
    ax2.set_ylabel('Truck')
    ax2.set_title('Original and Final Assignments')
    ax2.set_xticks(list(system.slots))
    ax2.set_xticklabels(
        [system.slots[slot_id].start_time for slot_id in system.slots],
        rotation=30,
    )
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # The stacked bars show how waiting and rescheduling costs form each total.
    ax3 = axes[1, 0]
    cost_comparison = pd.DataFrame(
        {
            'Method': ['Static', 'Dynamic'],
            'Waiting cost': [
                static_costs['waiting_cost'],
                dynamic_costs['waiting_cost'],
            ],
            'Rescheduling cost': [
                static_costs['rescheduling_cost'],
                dynamic_costs['rescheduling_cost'],
            ],
            'Total cost': [
                static_costs['total_cost'],
                dynamic_costs['total_cost'],
            ],
        }
    )
    x_positions = np.arange(len(cost_comparison))
    width = 0.55
    ax3.bar(
        x_positions,
        cost_comparison['Waiting cost'],
        width,
        label='Waiting cost',
        color='#1f77b4',
    )
    ax3.bar(
        x_positions,
        cost_comparison['Rescheduling cost'],
        width,
        bottom=cost_comparison['Waiting cost'],
        label='Rescheduling cost',
        color='#ff7f0e',
    )
    ax3.set_xlabel('Scheduling method')
    ax3.set_ylabel('Cost ($)')
    ax3.set_title('Static and Dynamic Scheduling Costs')
    ax3.set_xticks(x_positions)
    ax3.set_xticklabels(cost_comparison['Method'])
    ax3.legend()
    ax3.grid(True, axis='y', alpha=0.3)
    for position, total in enumerate(cost_comparison['Total cost']):
        ax3.text(position, total, f'${total:.0f}', ha='center', va='bottom')

    # Separate text rows avoid comparing unlike units on a shared bar-chart scale.
    ax4 = axes[1, 1]
    ax4.axis('off')
    rescheduling_count = sum(
        entry['old_slot'] != entry['new_slot']
        for entry in result['scheduling_log']
    )
    metric_lines = [
        ('Execution time', f"{result['execution_time'] * 1000:.3f} ms"),
        ('Trucks processed', f"{len(system.trucks)}"),
        ('Rescheduling decisions', f'{rescheduling_count}'),
        ('Cost reduction', f'{improvement_percent:.1f}%'),
    ]
    ax4.set_title('Run Metrics', pad=16)
    for index, (label, value) in enumerate(metric_lines):
        y_position = 0.82 - index * 0.2
        ax4.text(0.05, y_position, label, fontsize=12, transform=ax4.transAxes)
        ax4.text(
            0.95, y_position, value, fontsize=12, ha='right',
            transform=ax4.transAxes,
        )

    plt.tight_layout()
    plt.show()


create_comprehensive_visualizations(system, result, static_costs)

In [ ]:
def scalability_analysis() -> pd.DataFrame:
    """Measure runtime and shallow storage across generated problem sizes."""
    problem_sizes = [5, 10, 20, 50, 100]
    execution_times = []
    shallow_memory_kb = []

    for size in problem_sizes:
        trucks = {}
        slots = {}

        for index in range(size):
            truck_id = f'T{index + 1}'
            original_slot = (index % 10) + 1
            delay = np.random.randint(0, 60)
            urgency = np.random.uniform(0.5, 1.5)
            trucks[truck_id] = Truck(
                truck_id,
                original_slot,
                original_slot,
                int(delay),
                float(urgency),
                1.0,
            )

        slot_capacity = size // 5
        for slot_id in range(1, 11):
            hour = 8 + slot_id // 2
            minute = (slot_id % 2) * 30
            slots[slot_id] = TimeSlot(
                slot_id,
                f'{hour:02d}:{minute:02d}',
                slot_capacity,
            )

        for truck in trucks.values():
            if not slots[truck.original_slot].add_truck(truck.id):
                raise ValueError('Generated initial assignments exceed slot capacity.')

        test_system = SchedulingSystem(trucks, slots)

        start_time = time.perf_counter()
        priority_based_dynamic_scheduling(test_system)
        execution_times.append(time.perf_counter() - start_time)

        # This is a shallow estimate: referenced strings and nested object contents are excluded.
        shallow_bytes = sys.getsizeof(test_system.trucks)
        shallow_bytes += sys.getsizeof(test_system.slots)
        shallow_bytes += sum(sys.getsizeof(truck) for truck in trucks.values())
        shallow_bytes += sum(sys.getsizeof(slot) for slot in slots.values())
        shallow_bytes += sum(
            sys.getsizeof(slot.current_assignments) for slot in slots.values()
        )
        shallow_memory_kb.append(shallow_bytes / 1024)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    ax1.loglog(
        problem_sizes,
        execution_times,
        'o-',
        linewidth=2,
        markersize=8,
        label='Measured runtime',
    )
    ax1.set_xlabel('Number of trucks')
    ax1.set_ylabel('Execution time (seconds)')
    ax1.set_title('Measured Runtime by Problem Size')
    ax1.legend()
    ax1.grid(True, which='both', alpha=0.3)

    ax2.plot(
        problem_sizes,
        shallow_memory_kb,
        'o-',
        linewidth=2,
        markersize=8,
        color='#ff7f0e',
    )
    ax2.set_xlabel('Number of trucks')
    ax2.set_ylabel('Approximate shallow storage (KB)')
    ax2.set_title('Shallow Storage Estimate by Problem Size')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return pd.DataFrame(
        {
            'Problem size (trucks)': problem_sizes,
            'Execution time (seconds)': execution_times,
            'Approximate shallow storage (KB)': shallow_memory_kb,
        }
    )


print('Running scalability analysis...')
scalability_df = scalability_analysis()
print(scalability_df.round(4))

### Results analysis and interpretation

The code cells report results for the specific input data and generated scalability cases. The main example uses fixed truck and slot data. The scalability experiment draws delays and urgency weights from NumPy's random-number generator without setting a seed, so those generated inputs and their measurements can differ between runs. Runtime measurements also depend on the execution environment.

#### 1. Reading the scheduling decisions

The priority queue is constructed once from each truck's initial priority score. A high score determines earlier consideration; it does not itself mean that the truck must move. For each truck, `find_best_slot` compares the score of its current slot against feasible alternatives. The slot's occupancy is updated after each accepted move, so a slot that was available for one truck may be full when a later truck is considered.

The heuristic's decisions are local and sequential. A truck is not reconsidered after later assignments, and the method does not search all possible combinations of assignments. Consequently, the final schedule is a feasible result of the stated greedy procedure, not a proof of global optimality.

#### 2. Interpreting the cost comparison

Static cost assumes every truck keeps its original slot and bears its full expected delay. Dynamic cost uses the delay remaining after the model's later-slot absorption rule, plus a fixed charge for each changed appointment. Waiting cost and rescheduling cost should be read separately: a reduction in one component does not necessarily imply a reduction in total cost. The reported percentage compares total dynamic cost with total static cost for this example; it is not a general performance guarantee.

The slot score is a decision rule, not the objective function. Its availability bonus and load penalty can influence which feasible move is selected, while the cost report contains only the waiting and rescheduling terms defined in the model. This distinction is important when interpreting why the heuristic chose a particular slot.

#### 3. Interpreting the figures and scalability experiment

The priority chart displays the queue's initial scores, while the timeline shows original and final appointments. The stacked bars decompose total costs, and the run-metrics panel reports quantities with different units without placing them on a misleading shared scale.

The scalability experiment measures the implementation on generated cases; it does not establish a hardware-independent runtime bound. Its storage figure is a shallow Python-object estimate and excludes referenced strings and nested object contents. The asymptotic discussion above describes the operations in this implementation, whereas the plot shows measurements for only the tested sizes.

#### 4. Practical considerations

In an operational system, delay estimates, urgency weights, slot capacities, cost rates, and the assumed delay absorption per later slot would need to reflect the terminal's processes. The current notebook represents a single update snapshot and a simplified cost model. It does not model uncertainty in future arrivals, service-time variation, driver communication, or the effects of appointment changes on other terminal resources.